In [ ]:
# imports
import numpy as np
import pandas as pd
import sys

import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
plt.style.use('scifigs.mplstyle')
SAVE_FIG = False

sys.path.append('helpers/pcca_fa/')
from dual_pfc_funcs import load_dict, getParams, get_top_angle
import helpers.pcca_fa.pcca_fa_mdl as pf
color_map = getParams()['color_map']

In [ ]:
fname = 'preprocessed_data/simdataset_varyTheta.pkl'
dat = load_dict(fname)
n_boots = dat['n_boots']
thetas = dat['thetas']
theta_list = []
for theta in thetas:
    for j in range(n_boots):
        theta_list.append(theta)
# GT SV:
sv_across = dat['sv'][0]
sv_within = dat['sv'][1]
ind_var = 100 - (sv_across + sv_within)

In [ ]:
df = pd.DataFrame(columns=['Theta','GT-Theta_X1','GT-Theta_X2',
                           'GT-SvW1','GT-SvW2','GT-SvL1','GT-SvL2','GT-IndV_X1','GT-IndV_X2',
                           'Est-SvW1','Est-SvW2','Est-SvL1','Est-SvL2','Est-IndV_X1','Est-IndV_X2',
                           'Error-SvW1','Error-SvW2','Error-SvL1','Error-SvL2','Error-IndV_X1','Error-IndV_X2',
                           'Est-Theta_X1','Est-Theta_X2','Error-Theta_X1','Error-Theta_X2'])

for (gt,est,theta) in zip(dat['sim_params'],dat['est_params'],theta_list):
    mdl = pf.pcca_fa()
    mdl.set_params(gt)
    psv_gt = mdl.compute_psv().copy()
    gt_thetax1,gt_thetax2 = get_top_angle(gt)

    gt_ind_var_x1 = np.mean(psv_gt['ind_var_x1'])
    gt_ind_var_x2 = np.mean(psv_gt['ind_var_x2'])

    mdl = pf.pcca_fa()
    mdl.set_params(est)
    psv_est = mdl.compute_psv().copy()
    est_thetax1,est_thetax2 = get_top_angle(est)

    est_ind_var_x1 = np.mean(psv_est['ind_var_x1'])
    est_ind_var_x2 = np.mean(psv_est['ind_var_x2'])

    df2 = {'Theta':theta,'GT-Theta_X1':gt_thetax1,'GT-Theta_X2':gt_thetax2,'Est-Theta_X1':est_thetax1,'Est-Theta_X2':est_thetax2,'Error-Theta_X1':est_thetax1-theta,'Error-Theta_X2':est_thetax2-theta,
           'GT-SvW1':psv_gt['avg_psv_W_1'],'GT-SvW2':psv_gt['avg_psv_W_2'],'GT-SvL1':psv_gt['avg_psv_L_1'],'GT-SvL2':psv_gt['avg_psv_L_2'],'GT-IndV_X1':gt_ind_var_x1,'GT-IndV_X2':gt_ind_var_x2,
           'Est-SvW1':psv_est['avg_psv_W_1'],'Est-SvW2':psv_est['avg_psv_W_2'],'Est-SvL1':psv_est['avg_psv_L_1'],'Est-SvL2':psv_est['avg_psv_L_2'],'Est-IndV_X1':est_ind_var_x1,'Est-IndV_X2':est_ind_var_x2,
           'Error-SvW1':psv_est['avg_psv_W_1']-psv_gt['avg_psv_W_1'],'Error-SvW2':psv_est['avg_psv_W_2']-psv_gt['avg_psv_W_2'],'Error-SvL1':psv_est['avg_psv_L_1']-psv_gt['avg_psv_L_1'],'Error-SvL2':psv_est['avg_psv_L_2']-psv_gt['avg_psv_L_2'],'Error-IndV_X1':est_ind_var_x1-gt_ind_var_x1,'Error-IndV_X2':est_ind_var_x2-gt_ind_var_x2}
    df.loc[len(df)] = df2
# df

In [ ]:
ms = 1 # marker size
lims = 7
ticks = np.arange(-lims,lims,3)+1
fig,ax = plt.subplots(1,3,sharex=True,sharey=True,figsize=(9,5))

fig.supxlabel('error in within-area %sv',color=color_map['within'],y=0.15)
fig.supylabel('error in across-area %sv',color=color_map['across'],x=0.05)

ax[0].scatter(0,0,s=100,color='gray', marker='o')
ax[1].scatter(0,0,s=100,color='gray', marker='o')
ax[2].scatter(0,0,s=100,color='gray', marker='o')

ax[0].plot(df[df['Theta']==1]['Error-SvL1'], df[df['Theta']==1]['Error-SvW1'], color='black', ms=ms, ls='',marker='o')
ax[1].plot(df[df['Theta']==20]['Error-SvL1'], df[df['Theta']==20]['Error-SvW1'], color='black', ms=ms, ls='',marker='o')
ax[2].plot(df[df['Theta']==80]['Error-SvL1'], df[df['Theta']==80]['Error-SvW1'], color='black', ms=ms, ls='',marker='o')

ax[0].plot(df[df['Theta']==1]['Error-SvL2'], df[df['Theta']==1]['Error-SvW2'], color='black', ms=ms, ls='',marker='o')
ax[1].plot(df[df['Theta']==20]['Error-SvL2'], df[df['Theta']==20]['Error-SvW2'], color='black', ms=ms, ls='',marker='o')
ax[2].plot(df[df['Theta']==80]['Error-SvL2'], df[df['Theta']==80]['Error-SvW2'], color='black', ms=ms, ls='',marker='o')

ax[0].set_title(r'$\theta_{sim} = 1$ degree')
ax[1].set_title(r'$\theta_{sim} = 20$ degrees')
ax[2].set_title(r'$\theta_{sim} = 80$ degrees')

ax[0].set_xlim([-lims,lims]),ax[0].set_ylim([-lims,lims])
ax[0].set_xticks(ticks),ax[0].set_yticks(ticks)

ax[0].set_aspect('equal')
ax[1].set_aspect('equal')
ax[2].set_aspect('equal')

if SAVE_FIG:
    pdf = PdfPages('figs/sv_error_ex_theta.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
# next panel - mean and std of psv
sv_across_mean, sv_within_mean, ind_mean = [], [], []
sv_across_std, sv_within_std, ind_std = [], [], []
for theta in thetas:
    filt = df['Theta'] == theta

    # calculate error in estimated shared variance
    curr_across = np.concatenate((df[filt]['Error-SvW1'],df[filt]['Error-SvW2']))
    curr_within = np.concatenate((df[filt]['Error-SvL1'],df[filt]['Error-SvL2']))
    curr_ind = np.concatenate((df[filt]['Error-IndV_X1'],df[filt]['Error-IndV_X2']))

    sv_across_mean.append(np.mean(curr_across))
    sv_within_mean.append(np.mean(curr_within))
    ind_mean.append(np.mean(curr_ind))
    sv_across_std.append(np.std(curr_across))
    sv_within_std.append(np.std(curr_within))
    ind_std.append(np.std(curr_ind))

fig,ax = plt.subplots(3,1,sharex=True,tight_layout=True,sharey=True,figsize=(5,9))

fig.supxlabel(r'$\theta_{sim}$ (degrees)')

# ground truth:
ax[0].plot([0,90],[0,0],'--', color='gray') # global
ax[1].plot([0,90],[0,0],'--', color='gray') # local
ax[2].plot([0,90],[0,0],'--', color='gray') # ind

# ests:
ax[0].errorbar(thetas, sv_across_mean, yerr=sv_across_std, fmt='-o', color=color_map['across'], ms=4)
ax[1].errorbar(thetas, sv_within_mean, yerr=sv_within_std, fmt='-o', color=color_map['within'], ms=4)
ax[2].errorbar(thetas, ind_mean, yerr=ind_std, fmt='-o', color='black', ms=4)

ax[0].set_ylabel('error in across-area %sv', color=color_map['across'])
ax[1].set_ylabel('error in within-area %sv', color=color_map['within'])
ax[2].set_ylabel('error in % independent variance', color='black')

ax[0].set_xlim([-5,95])
ax[0].set_ylim([-5,5])
fig.show()

if SAVE_FIG:
    pdf = PdfPages('figs/sv_error_all_theta.pdf')
    pdf.savefig(fig)
    pdf.close()
else:
    fig.show()

In [ ]:
df.groupby(['Theta']).agg({'Error-Theta_X1': ['count', 'mean', np.std],
                           'Error-Theta_X2': ['count', 'mean', np.std]})